#### Pydantic

Pydantic 模型提供自动验证

In [6]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movie's rating out of 10")

model = init_chat_model('deepseek-chat')

In [10]:
model_with_structure = model.with_structured_output(Movie)
response1 = model_with_structure.invoke("Provide details about the movie Inception")

In [11]:
type(response1), response1

(__main__.Movie,
 Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8))

获取解析后的输出和原始 AI 消息:

In [12]:
model_with_structure = model.with_structured_output(Movie, include_raw=True)
response2 = model_with_structure.invoke("Provide details about the movie Inception")

In [13]:
response2

{'raw': AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 89, 'prompt_tokens': 390, 'total_tokens': 479, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384}, 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 6}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache', 'id': '3d033581-6631-4037-95bd-e9f6e2ab3dfb', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d4382-6fa2-7b31-b70c-5a14e1c424ed-0', tool_calls=[{'name': 'Movie', 'args': {'title': 'Inception', 'year': 2010, 'director': 'Christopher Nolan', 'rating': 8.8}, 'id': 'call_00_0hag1BMidRp9ynZrbXN1i9Bz', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 390, 'output_tokens': 89, 'total_tokens': 479, 'input_token_details': {'cache_read': 384}, 'output_token_details': {}}),
 'parsed': Mo

#### TypedDict

In [14]:
from typing import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

In [15]:
model_with_structure = model.with_structured_output(MovieDict)
response3 = model_with_structure.invoke("Provide details about the movie Inception")

In [16]:
response3

{'title': 'Inception',
 'year': 2010,
 'director': 'Christopher Nolan',
 'rating': 8.8}

#### JSON Schema

method 参数：

- `json_schema` ：使用提供商提供的专用结构化输出功能。

- `function_calling` ：通过强制按给定模式调用工具来导出结构化输出。

In [17]:
import json

json_schema = {
    "title": "Movie",
    "description": "A movie with details",
    "type": "object",
    "properties": {
        "title": {
            "type": "string",
            "description": "The title of the movie"
        },
        "year": {
            "type": "integer",
            "description": "The year the movie was released"
        },
        "director": {
            "type": "string",
            "description": "The director of the movie"
        },
        "rating": {
            "type": "number",
            "description": "The movie's rating out of 10"
        }
    },
    "required": ["title", "year", "director", "rating"]
}

In [18]:
model_with_structure = model.with_structured_output(
    json_schema,
    method="json_schema",
)
response4 = model_with_structure.invoke("Provide details about the movie Inception")

In [19]:
response4

{'title': 'Inception',
 'year': 2010,
 'director': 'Christopher Nolan',
 'rating': 8.8}